# 🤖 Scikit-Learn Integration Example

This notebook demonstrates how to integrate `mlprep` into a Python-based machine learning workflow.

## Workflow
1. **Feature Engineering with mlprep**: Scalable, consistent preprocessing using `mlprep` binary.
2. **Training with Scikit-Learn**: Loading the processed Parquet file for model training.

## 🔧 Setup

Install `mlprep-rust` and `scikit-learn` packages.

In [ ]:
!pip install mlprep-rust scikit-learn -q

## 📁 Generate Sample Data

Generate a classification dataset with numeric features.

In [ ]:
import pandas as pd
import numpy as np

def generate_data():
    np.random.seed(42)
    n_rows = 200
    df = pd.DataFrame({
        'feature1': np.random.normal(0, 1, n_rows),
        'feature2': np.random.normal(5, 2, n_rows),
        'feature3': np.random.choice(['A', 'B'], n_rows),  # Categorical feature
        'target': np.random.randint(0, 2, n_rows)
    })
    df.to_csv('raw_data.csv', index=False)
    print("Generated raw_data.csv with 200 rows")
    return df

df = generate_data()
print("\n📊 Data Sample:")
print(df.head(10))
print("\n📈 Statistics:")
print(df.describe())

## 📝 Create Pipeline Configuration

Create pipeline for feature preprocessing.

In [ ]:
pipeline_yaml = """
name: sklearn_prep
inputs:
  - path: raw_data.csv
    format: csv

steps:
  - type: select
    columns: [feature1, feature2, feature3, target]
  - type: features
    config:
      - columns: [feature1, feature2]
        method: standard

outputs:
  - path: processed_train.parquet
    format: parquet
"""

with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml.strip())

print("Created pipeline.yaml")
print(pipeline_yaml)

## 🚀 Run mlprep Pipeline

Execute the preprocessing pipeline.

In [ ]:
!mlprep run pipeline.yaml

## 🤖 Train Scikit-Learn Model

Load the processed data and train a Logistic Regression model.

In [ ]:
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

if os.path.exists('processed_train.parquet'):
    print("✅ Loading processed data...")
    df = pd.read_parquet('processed_train.parquet')
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Prepare features and target
    X = df[['feature1', 'feature2']]
    y = df['target']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f"\nTraining samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")
    
    # Train model
    print("\n🎯 Training Logistic Regression...")
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    
    print(f"\n📊 Results:")
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
else:
    print("❌ processed_train.parquet not found. Run the mlprep pipeline first.")

## 📊 Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

if 'test_preds' in dir():
    print("📈 Classification Report:")
    print(classification_report(y_test, test_preds))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, test_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

## 📊 Summary

Full workflow summary.

In [ ]:
print("🔄 ML Pipeline Workflow Summary")
print("="*50)
print("\n1️⃣ Data Generation:")
print(f"   - Generated 200 samples with 4 columns")
print(f"   - Features: feature1 (normal), feature2 (normal), feature3 (categorical), target")

print("\n2️⃣ Preprocessing with mlprep:")
print("   - Selected columns: feature1, feature2, feature3, target")
print("   - Standard scaling on feature1, feature2")
print("   - Output: processed_train.parquet")

print("\n3️⃣ Model Training with Scikit-Learn:")
print("   - Algorithm: Logistic Regression")
print("   - Train/Test split: 80/20")
if 'test_acc' in dir():
    print(f"   - Test Accuracy: {test_acc:.4f}")

print("\n✅ Pipeline completed successfully!")